# タクシー運賃予測のための最初の BigQuery ML モデル

**学習目標**
  * 適切な BigQuery ML モデル タイプを選択し、オプションを指定する
  * ML モデルのパフォーマンスを評価する
  * データ品質のクリーンアップを通じてモデルのパフォーマンスを向上させる
  * SQL を使用してディープ ニューラル ネットワーク（DNN）を作成する

## 概要
このノートブックでは、BigQuery ML を使用してタクシー運賃予測のための最初のモデルを構築します。BigQuery ML は、大規模な構造化データおよび半構造化データに対して ML モデルを迅速に構築する方法を提供します。まず、BigQuery で作成するすべてのモデルを保持するためのデータセットの作成から始めます。

### 環境変数の設定

In [ ]:
PROJECT = !gcloud config get-value project
PROJECT = PROJECT[0]
BUCKET = PROJECT
REGION = "us-central1"

%env PROJECT=$PROJECT
%env BUCKET=$BUCKET
%env REGION=$REGION

%load_ext google.cloud.bigquery

## BigQuery データセットと Google Cloud Storage バケットの作成

BigQuery データセットは、テーブル、ビュー、および BigQuery ML で構築されたモデル用のコンテナです。以前のラボでまだ作成していない場合は、__serverlessml__ という名前のデータセットを作成しましょう。プロジェクト用の GCS バケットについても同様に行います。

In [ ]:
%%bash

# serverlessml 用の BigQuery データセットが存在しない場合は作成
datasetexists=$(bq ls -d | grep -w serverlessml)

if [ -n "$datasetexists" ]; then
    echo -e "BigQuery dataset already exists, let's not recreate it."
else
    echo "Creating BigQuery dataset titled: serverlessml"

    bq --location=US mk --dataset \
        --description 'Taxi Fare' \
        $PROJECT:serverlessml
    echo "\nHere are your current datasets:"
    bq ls
fi    

# GCS バケットがまだ存在しない場合は作成...
exists=$(gcloud storage ls --buckets | grep -w gs://${BUCKET}/)

if [ -n "$exists" ]; then
    echo -e "Bucket exists, let's not recreate it."
else
    echo "Creating a new GCS bucket."
    gcloud storage buckets create --location=${REGION} gs://${BUCKET}
    echo "\nHere are your current buckets:"
    gcloud storage ls
fi

## モデル 1: 生データ

生データのみを使用してモデルを構築してみましょう。あまり良い結果にはなりませんが、実際にこれを体験してみることも重要です。

モデルのトレーニングには 1 分程度かかります。機械学習としては非常に高速です。

In [ ]:
%%bigquery
CREATE OR REPLACE MODEL
    serverlessml.model1_rawdata

# TODO 1: 予測に適した ML model_type を選択:
# 例: 線形回帰 (linear_reg) またはロジスティック回帰 (logistic_reg)
# 下の行に適切な ML OPTIONS() を入力:

SELECT
    (tolls_amount + fare_amount) AS fare_amount,
    pickup_longitude AS pickuplon,
    pickup_latitude AS pickuplat,
    dropoff_longitude AS dropofflon,
    dropoff_latitude AS dropofflat,
    passenger_count * 1.0 AS passengers
FROM
    `nyc-tlc.yellow.trips`
WHERE
    ABS(MOD(FARM_FINGERPRINT(CAST(pickup_datetime AS STRING)), 100000)) = 1

トレーニングが完了したら、[BigQuery Cloud Console](https://console.cloud.google.com/bigquery) にアクセスしてトレーニングされたモデルを確認してください。その後、このノートブックに戻ります。

BigQuery は提供されたデータを自動的に分割し、データの一部のみでトレーニングを行い、残りを評価に使用したことに注目してください。そのホールドアウトされたデータに対する評価統計を確認できます。

In [ ]:
%%bigquery
# TODO 2: 新しくトレーニングしたモデルを評価するコマンドを指定
SELECT * FROM

関心のある誤差、すなわち二乗平均平方根誤差（RMSE）のみをレポートしてみましょう。

In [ ]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL serverlessml.model1_rawdata)

良い結果にはならないとお伝えしたとおりです。ヒューリスティックによる結果が 8.13 であり、目標値が 6 ドルであったことを思い出してください。

誤差は評価を行うデータセットに依存することに注意してください。
独自のホールドアウトされたベンチマーク / テスト データセットでモデルを評価することもできますが、これを習慣にするべきではありません（ベンチマーク データセットは最終評価用として保持し、開発の途中で判断材料として使い続けるべきではありません。そうしてしまうと、テスト データセットが真に独立したものにならなくなります）。

In [ ]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL serverlessml.model1_rawdata, (
    SELECT
        (tolls_amount + fare_amount) AS fare_amount,
        pickup_longitude AS pickuplon,
        pickup_latitude AS pickuplat,
        dropoff_longitude AS dropofflon,
        dropoff_latitude AS dropofflat,
        passenger_count * 1.0 AS passengers # 小数として処理
    FROM
        `nyc-tlc.yellow.trips`
    WHERE
        ABS(MOD(FARM_FINGERPRINT(CAST(pickup_datetime AS STRING)), 100000)) = 2
        # 後の TODO 3 の一部としての追加フィルタ用プレースホルダ
    ))

上記からの RMSE はいくつでしたか？

TODO 3: 前のクエリの WHERE 句内に以下のフィルタを適用してください。パフォーマンスは向上しますか？それはなぜですか？

```sql
    AND trip_distance > 0
    AND fare_amount >= 2.5
    AND pickup_longitude > -78
    AND pickup_longitude < -70
    AND dropoff_longitude > -78
    AND dropoff_longitude < -70
    AND pickup_latitude > 37
    AND pickup_latitude < 45
    AND dropoff_latitude > 37
    AND dropoff_latitude < 45
    AND passenger_count > 0
```

## モデル 2: データのクリーンアップの適用

前のラボでいくつかのデータ クリーンアップを行ったことを思い出してください。トレーニングの前にそれらを実行しましょう。

これはこのノートブックで頻繁に必要となるデータセットであるため、まず抽出しておきます。

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE
    serverlessml.cleaned_training_data AS

SELECT
    (tolls_amount + fare_amount) AS fare_amount,
    pickup_longitude AS pickuplon,
    pickup_latitude AS pickuplat,
    dropoff_longitude AS dropofflon,
    dropoff_latitude AS dropofflat,
    passenger_count*1.0 AS passengers
FROM
    `nyc-tlc.yellow.trips`
WHERE
    ABS(MOD(FARM_FINGERPRINT(CAST(pickup_datetime AS STRING)), 100000)) = 1
    AND trip_distance > 0
    AND fare_amount >= 2.5
    AND pickup_longitude > -78
    AND pickup_longitude < -70
    AND dropoff_longitude > -78
    AND dropoff_longitude < -70
    AND pickup_latitude > 37
    AND pickup_latitude < 45
    AND dropoff_latitude > 37
    AND dropoff_latitude < 45
    AND passenger_count > 0

In [ ]:
%%bigquery
-- LIMIT 0 は無料のクエリであり、テーブルが存在することを確認できます。
SELECT * FROM serverlessml.cleaned_training_data
LIMIT 0

In [ ]:
%%bigquery
CREATE OR REPLACE MODEL
    serverlessml.model2_cleanup

OPTIONS(input_label_cols=['fare_amount'],
        model_type='linear_reg') AS

SELECT
    *
FROM
    serverlessml.cleaned_training_data

In [ ]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL serverlessml.model2_cleanup)

## モデル 3: より洗練されたモデル

より洗練されたモデルを試すとどうなるでしょうか？BigQuery でディープ ニューラル ネットワーク（DNN）を試してみましょう。

### DNN
DNN を作成するには、model_type に __dnn_regressor__ を指定し、隠れ層（hidden layers）を追加するだけです。

In [ ]:
%%bigquery
-- このトレーニングには約 15 分かかります。
CREATE OR REPLACE MODEL
    serverlessml.model3b_dnn

# TODO 4a: DNN とラベル フィールドに適切な BigQuery ML モデル タイプを選択
# 選択肢: dnn_regressor, linear_reg, logistic_reg
OPTIONS() AS

SELECT
    *
FROM
    serverlessml.cleaned_training_data

In [ ]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
FROM
    ML.EVALUATE(MODEL serverlessml.model3b_dnn)

素晴らしい結果です。

## ベンチマーク データセットでの DNN の評価

評価には同じ検証データセットを使用しましょう。評価指標はデータセットに依存することを忘れないでください。同じホールドアウト データで実行しない限り、2 つのモデルを比較することはできません。

In [ ]:
%%bigquery
SELECT
    SQRT(mean_squared_error) AS rmse
# TODO 4b: ML モデルのパフォーマンスを確認するためのコマンドは何ですか？
# ML.What? 
FROM
    ML.WHATCOMMAND(MODEL serverlessml.model3b_dnn, (
    SELECT
        (tolls_amount + fare_amount) AS fare_amount,
        pickup_datetime,
        pickup_longitude AS pickuplon,
        pickup_latitude AS pickuplat,
        dropoff_longitude AS dropofflon,
        dropoff_latitude AS dropofflat,
        passenger_count * 1.0 AS passengers,
        'unused' AS key
    FROM
        `nyc-tlc.yellow.trips`
    WHERE
        ABS(MOD(FARM_FINGERPRINT(CAST(pickup_datetime AS STRING)), 10000)) = 2
        AND trip_distance > 0
        AND fare_amount >= 2.5
        AND pickup_longitude > -78
        AND pickup_longitude < -70
        AND dropoff_longitude > -78
        AND dropoff_longitude < -70
        AND pickup_latitude > 37
        AND pickup_latitude < 45
        AND dropoff_latitude > 37
        AND dropoff_latitude < 45
        AND passenger_count > 0
    ))

素晴らしいですね。この一連のノートブックの後半では 4 ドル未満まで改善されますが、わずかな作業でこの精度が得られるのは非常に優れています。

このノートブックでは、BigQuery ML を使用して ML モデルを迅速に構築する方法を紹介しました。さまざまな特徴量エンジニアリングを試したいときに、再び BigQuery ML に戻ってきます。BigQuery ML のスピードは開発において非常に魅力的です。

Copyright 2021 Google Inc.
Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at
http://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.